# The 2026 Sentiment Gap — analysis notebook

This notebook runs the full pipeline of the study, step by step. It shows the
same numbers that `build_report.py` puts in `perceptual_bias_report.pdf`.

This notebook uses ASD-STE100 Simplified Technical English.

**How to read this notebook.** Each step calls one public function of one
module. Each function returns an object. The object holds the result and also a
record of the data that the function removed or approximated. The notebook
prints that record. A row that the pipeline removes has a name and a cause.

**The network.** The notebook needs no network. FRED is optional. Without a
FRED key the notebook builds the panel with the documented approximation of the
price level, and it marks each result that FRED would have given. The notebook
does not stop.

**What this notebook does not have.** Hypothesis B is not tested. The GDELT
media corpus is still downloading. Step 6 gives the state of the corpus and the
exact list of results that the corpus blocks.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))

import build_report as br
from perceptual_bias import baseline_model as bm
from perceptual_bias import causal_econometrics as ce
from perceptual_bias import data_loader as dl
from perceptual_bias import survey_diagnostics as sd
from perceptual_bias import table_generator as tg

pd.set_option("display.width", 110)
pd.set_option("display.max_columns", 20)
print("repository root:", ROOT)

repository root: /Users/paramshah/projects/perceptual-bias


## Step 1. The panel

`build_monthly_panel` reads the issued CSV file and, if a key is available,
FRED. It returns the frame and a `PanelReport`. The report says what the build
got and what it approximated.

In [2]:
panel, panel_report = dl.build_monthly_panel(ROOT / "data" / "raw" / "Copy_of_Perceptual_Bias_Data.csv")
print(panel_report.summary())
print()
for note in panel_report.notes:
    print(" -", note)
FRED_AVAILABLE = panel_report.fred_available
print()
print("FRED available in this run:", FRED_AVAILABLE)
panel.tail(6)

586 monthly rows, 1978-01 to 2026-10 | core: unemployment, sentiment, pce_yoy, real_gdp_qoq | FRED: unavailable | price level: pce_level_approx

 - real_gdp_qoq: quarterly, expanded across its own quarter's three months (contemporaneous, not real-time)
 - pce_level_approx: APPROXIMATION accumulated from published YoY rates in logs (2020 mean = 100); PCEPI unavailable

FRED available in this run: False


,unemployment,sentiment,pce_yoy,real_gdp_qoq,pce_level_approx
date,,,,,
2026-05-31,4.3,44.8,3.16171,2.2,122.974607
2026-06-30,4.2,49.5,3.04123,2.2,123.286664
2026-07-31,4.1,55.2,2.98347,3.7,123.593564
2026-08-31,4.1,51.7,3.00759,3.7,123.903718
2026-09-30,4.2,48.1,NaN,3.7,NaN
2026-10-31,NaN,46.3,NaN,NaN,NaN


Real GDP growth arrives each quarter. The loader puts each quarterly value in
the three months of its own quarter. This alignment is contemporaneous. It is
not real time. A forward fill gives a different conclusion: the GDP coefficient
then becomes -0.037 with a p-value of 0.518.

## Step 2. The baseline model

The baseline puts sentiment on the unemployment rate, the inflation rate and
real GDP growth. The window is 2010-01 to 2024-12. Inference uses Newey-West
HAC standard errors with 12 lags, because monthly macro data has strong
autocorrelation.

In [3]:
fit = bm.fit_baseline(dl.training_slice(panel))
print(fit.equation())
print()
print(f"n          = {fit.n_train}")
print(f"R-squared  = {fit.r_squared:.3f}")
print(f"adj R2     = {fit.adj_r_squared:.3f}")
print(f"RMSE       = {fit.rmse_in_sample:.2f}")
print(f"AIC        = {fit.aic:.1f}")
print()
for note in fit.notes:
    print(" -", note)

sentiment = 127.56 - 4.28*unemployment - 9.77*pce_yoy - 0.16*real_gdp_qoq

n          = 180
R-squared  = 0.766
adj R2     = 0.762
RMSE       = 6.20
AIC        = 1175.5

 - ridge alpha=0.01 chosen by 5-fold forward-chaining CV
 - OLS inference uses Newey-West HAC errors, maxlags=12


**A warning, not a result.** The GDP coefficient is negative and significant.
More growth must not give less sentiment. In this window high growth follows a
shock, in 2010 and in 2021, and sentiment is still low at those times. The
coefficient measures the shock. Report this as a warning about the
specification.

## Step 3. The Sentiment Gap

The gap is the observed sentiment minus the predicted sentiment, on data that
the model never saw. The headline is the **count**, not the size: the share of
months outside the model's own 95% prediction interval.

In [4]:
gaps, gap_report = bm.sentiment_gap(fit, dl.holdout_slice(panel))
stats = bm.gap_summary(gaps)
print(gap_report.summary())
for reason in gap_report.dropped_reasons:
    print("   dropped:", reason)
print()
print(stats.to_string())

19 rows predicted; dropped 2025-10, 2026-09, 2026-10
   dropped: 2025-10: missing unemployment
   dropped: 2026-09: missing pce_yoy
   dropped: 2026-10: missing unemployment, pce_yoy, real_gdp_qoq

n_months                  19.0
n_tested                  19.0
mean_gap            -25.242516
median_gap          -25.502649
worst_gap            -33.13259
worst_month            2026-05
worst_observed            44.8
worst_predicted       77.93259
months_outside_pi         18.0
share_outside_pi      0.947368


In [5]:
outside = int(stats["months_outside_pi"])
tested = int(stats["n_tested"])
display(Markdown(
    f"**{outside} of {tested} out-of-sample months fall outside the 95% prediction "
    f"interval, and every month falls on the same side.** A correct model puts "
    f"about one month in twenty outside that interval. The mean gap is "
    f"{stats['mean_gap']:.2f} and the worst gap is {stats['worst_gap']:.2f} at "
    f"{stats['worst_month']}. The count is the evidence of model breakdown. The "
    f"size alone is not."
))

**18 of 19 out-of-sample months fall outside the 95% prediction interval, and every month falls on the same side.** A correct model puts about one month in twenty outside that interval. The mean gap is -25.24 and the worst gap is -33.13 at 2026-05. The count is the evidence of model breakdown. The size alone is not.

### Table 1

`table_generator` renders one `Table` object twice. The Markdown below and the
LaTeX in the PDF come from the same object, so they cannot drift.

In [6]:
table1 = br.build_table1(br.BuildContext(
    fit=fit, gaps=gaps, gap_report=gap_report, gap_stats=stats,
))
display(Markdown(table1.to_markdown()))

**Table 1: Historical macro OLS baseline (2010-2024) and the 2025-2026 Sentiment Gap**

*Panel A. Baseline OLS, monthly, HAC(12) standard errors*

| Regressor | Coefficient | HAC SE | t | p |
|:---|---:|---:|---:|---:|
| const | 127.558*** | 4.449 | 28.67 | <0.001 |
| unemployment | -4.276*** | 0.522 | -8.19 | <0.001 |
| pce_yoy | -9.770*** | 0.914 | -10.69 | <0.001 |
| real_gdp_qoq | -0.159** | 0.077 | -2.07 | 0.039 |

*Panel B. Fit statistics*

| Statistic | Value |
|:---|---:|
| R^2 | 0.766 |
| Adjusted R^2 | 0.762 |
| RMSE (in sample) | 6.20 |
| AIC | 1175.5 |
| Training window | 2010-01 to 2024-12 |
| Observations | 180 |

*Panel C. Out-of-sample Sentiment Gap (observed - predicted)*

| Month | Observed | Predicted | Gap | Outside 95% PI |
|:---|---:|---:|---:|---:|
| 2025-01 | 71.70 | 82.97 | -11.27 | no |
| 2025-02 | 64.70 | 82.00 | -17.30 | yes |
| 2025-03 | 57.00 | 84.54 | -27.54 | yes |
| 2025-04 | 52.20 | 82.82 | -30.62 | yes |
| 2025-05 | 52.20 | 81.30 | -29.10 | yes |
| 2025-06 | 60.70 | 81.79 | -21.09 | yes |
| 2025-07 | 61.70 | 80.79 | -19.09 | yes |
| 2025-08 | 58.20 | 80.70 | -22.50 | yes |
| 2025-09 | 55.10 | 80.60 | -25.50 | yes |
| 2025-11 | 51.00 | 81.70 | -30.70 | yes |
| 2025-12 | 52.90 | 80.65 | -27.75 | yes |
| 2026-01 | 56.40 | 80.07 | -23.67 | yes |
| 2026-02 | 56.60 | 79.44 | -22.84 | yes |
| 2026-03 | 53.30 | 78.44 | -25.14 | yes |
| 2026-04 | 49.80 | 78.69 | -28.89 | yes |
| 2026-05 | 44.80 | 77.93 | -33.13 | yes |
| 2026-06 | 49.50 | 79.54 | -30.04 | yes |
| 2026-07 | 55.20 | 80.29 | -25.09 | yes |
| 2026-08 | 51.70 | 80.06 | -28.36 | yes |
| **Summary** |  |  |  |  |
| Mean gap |  |  | -25.24 |  |
| Median gap |  |  | -25.50 |  |
| Worst month (2026-05) | 44.80 | 77.93 | -33.13 |  |
| Months outside 95% PI |  |  | 18 of 19 | 94.7% |

*Notes:* Dependent variable: sentiment. n = 180 training months.  
Standard errors are Newey-West HAC with maxlags=12.  
Significance: * p<0.10, ** p<0.05, *** p<0.01.  
Gap is evaluated out of sample; the prediction interval is a 95% interval for an individual month, so it includes residual variance.  
Months with no computable gap: 2025-10: missing unemployment; 2026-09: missing pce_yoy; 2026-10: missing unemployment, pce_yoy, real_gdp_qoq.  

### The two figures

The figures use the project palette. Each chart has one y-axis. A second axis
would invent a relation between two scales.

In [7]:
ctx = br.gather(ROOT / "data" / "raw" / "Copy_of_Perceptual_Bias_Data.csv")
ctx.figures = br.make_figures(ctx)
for key in ("gap_fan", "gap_bars"):
    figure = ctx.figures.get(key)
    if figure is None:
        print(f"{key}: not drawn, because the gap frame is absent")
        continue
    display(Markdown(f"![{figure.caption}]({figure.path.as_posix()})"))
    display(Markdown(f"*{figure.caption}*"))

FRED is not configured in this environment, so this build cannot fetch an independent comparison survey


![Figure 1. Observed sentiment against the Model 1 prediction, with the 95% prediction interval for a single month. The observed series stays below the interval in every month but one.](/Users/paramshah/projects/perceptual-bias/outputs/figures/fig1_observed_vs_predicted.png)

*Figure 1. Observed sentiment against the Model 1 prediction, with the 95% prediction interval for a single month. The observed series stays below the interval in every month but one.*

![Figure 2. The Sentiment Gap in each out-of-sample month. Every month is negative. The grey bar is the one month inside the 95% prediction interval.](/Users/paramshah/projects/perceptual-bias/outputs/figures/fig2_monthly_gap.png)

*Figure 2. The Sentiment Gap in each out-of-sample month. Every month is negative. The grey bar is the one month inside the 95% prediction interval.*

## Step 4. Hypothesis A. Economic pressure that the baseline misses

Hypothesis A says that the baseline measures a rate of change, and that a
household feels a level. The test adds the cumulative PCE price level and the
30-year mortgage rate.

Model 2 needs the FRED series `PCEPI` and `MORTGAGE30US`. Without a key this
step reports Model 2 as not estimated. It does not substitute a number.

In [8]:
comparison = ce.compare_models(dl.training_slice(panel))
for name, model in comparison.models.items():
    if model.available:
        print(f"{name}: R2={model.r_squared:.3f} AIC={model.aic:.1f} n={model.n}")
    else:
        print(f"{name}: NOT ESTIMATED — {model.unavailable_reason}")
print()
print("common sample:", comparison.common_sample_n, "months")
print("evidence class:", comparison.evidence_class)

Model 1: R2=0.766 AIC=1175.5 n=180
Model 2: NOT ESTIMATED — Model 2 not estimated: columns absent from the panel: ['mortgage_30y']
Model 3: NOT ESTIMATED — Model 3 not estimated: columns absent from the panel: ['mortgage_30y', 'media_tone', 'topic_salience', 'political_attribution']

common sample: 180 months
evidence class: association (no identification claimed)


In [9]:
display(Markdown(br.build_table3(ctx).to_markdown()))

**Table 3: Model comparison matrix**

| Statistic | Model 1 | Model 2 | Model 3 (n/a) |
|:---|---:|---:|---:|
| R^2 | 0.766 | 0.835 | n/a |
| Adjusted R^2 | 0.762 | n/a | n/a |
| RMSE | 6.20 | n/a | n/a |
| AIC | 1175.5 | 1117.4 | n/a |
| Observations | 180 | n/a | n/a |
| Mean out-of-sample gap | -25.24 | -17.40 | n/a |

*Notes:* Model 1 is the macro baseline. Model 2 adds the Hypothesis A price level and mortgage rate. Model 3 adds the Hypothesis B media series. The fit statistics are the training window 2010-01 to 2024-12.  
Standard errors are Newey-West HAC with maxlags=12.  
Significance: * p<0.10, ** p<0.05, *** p<0.01.  
Model 3 is not available: the media regressors do not exist yet; the GDELT corpus is still downloading: 4 of 232 days are cached, so no TF-IDF matrix and no SVD basis exist yet.  
Model 2 was NOT estimated in this build. FRED is not configured here, so the mortgage rate and the PCE price level are absent. Its R-squared, AIC and mean gap are the recorded values from BRAINSTORMING.md paragraph 2.1. Its adjusted R-squared, RMSE and observation count are not recorded, so they print as n/a. They are not zero and not small.  

**The honest statement.** The Hypothesis A variables absorb about one third of
the gap. The mortgage rate does the measurable work, with a coefficient of
-2.936 and p < 0.001. The price level is **not** significant, with a p-value of
0.310, and its p-value moves to 0.038 when the debt service ratio enters the
model. Its variance inflation factor is only 3.73, so collinearity alone does
not explain the result. **Give the price level no causal reading.**

Each of these numbers is a cross-checked constant. Step 8 prints the register
with the source of each one.

## Step 5. Hypothesis C. The survey method changed

The Michigan survey moved to the internet between 2024 and 2026. Test this
hypothesis first. Hypothesis A and Hypothesis B explain a true quantity. If
Hypothesis C is strong, then A and B compete to explain a number that is too
large.

In [10]:
sentiment = panel["sentiment"].dropna()
chow = sd.chow_test(sentiment)
print(f"Chow at {chow.breakpoint:%Y-%m}: F={chow.f_statistic:.2f} "
      f"p={chow.p_value:.3e} level shift={chow.level_shift:+.2f} "
      f"n={chow.n} k={chow.k}")
print()
print(chow.identification)

Chow at 2024-07: F=111.20 p=6.450e-24 level shift=-27.16 n=586 k=1

The Chow test asks whether the relationship's coefficients differ either side of a date the analyst supplied. It identifies a change at that date; it cannot identify its cause. Any event clustered near the survey's mode change — a policy shock, an election, a data revision — produces the same rejection, so a significant Chow statistic is consistent with Hypothesis C but is not evidence for it over any other same-dated explanation.


In [11]:
sup_wald = sd.sup_wald_test(sentiment, n_bootstrap=299)
print(f"sup-Wald break: {sup_wald.break_date:%Y-%m}  "
      f"stat={sup_wald.f_statistic:.1f}  bootstrap p={sup_wald.p_value:.4f}")
print(f"search window: {sup_wald.first_candidate:%Y-%m} to {sup_wald.last_candidate:%Y-%m}")
print(f"the 2024-07 transition is inside the search window: "
      f"{sup_wald.first_candidate <= sd.MICHIGAN_WEB_TRANSITION <= sup_wald.last_candidate}")

sup-Wald break: 2019-06  stat=139.0  bootstrap p=0.0033
search window: 1985-04 to 2019-06
the 2024-07 transition is inside the search window: False


The search cannot reach 2024-07. That date sits at about 96% of the sample, and
the trim removes it. "No break found there" is therefore **not** evidence
against Hypothesis C.

In [12]:
cross_survey = None
if FRED_AVAILABLE:
    try:
        series, notes = sd.load_comparison_series()
        usable, more = sd.verify_comparison_series(series, sentiment)
        for note in (*notes, *more):
            print(" -", note)
        if usable:
            cross_survey = sd.cross_survey_gap(panel, series)
        else:
            print("\nthe candidate did not pass verification; the leg stays untested")
    except sd.SurveyDiagnosticsError as exc:
        print("NOT AVAILABLE:", exc)
else:
    print("NOT AVAILABLE: FRED is not configured in this run, so this notebook")
    print("cannot fetch an independent comparison survey. The cross-survey leg")
    print("of Hypothesis C stays untested here. The recorded ratio is 0.258.")

NOT AVAILABLE: FRED is not configured in this run, so this notebook
cannot fetch an independent comparison survey. The cross-survey leg
of Hypothesis C stays untested here. The recorded ratio is 0.258.


In [13]:
verdict = sd.assess_hypothesis_c(chow=chow, sup_wald=sup_wald,
                                 cross_survey=cross_survey, gap=gaps["gap"])
ratio = cross_survey.gap_ratio if cross_survey is not None else None
shown = f"{ratio:.3f}" if ratio is not None else "0.258 (recorded, not measured in this run)"
display(Markdown(
    f"**The verdict rests on 0.008.** The cross-survey ratio is {shown}. The "
    f"threshold for the verdict 'supported' is {sd.SMALL_GAP_RATIO:.2f}. Always "
    f"print the ratio and the threshold together. Never print the word alone.\n\n"
    f"Module verdict: `{verdict.verdict}`."
))
for reason in verdict.reasons:
    print(" *", reason)

**The verdict rests on 0.008.** The cross-survey ratio is 0.258 (recorded, not measured in this run). The threshold for the verdict 'supported' is 0.25. Always print the ratio and the threshold together. Never print the word alone.

Module verdict: `partially_supported`.

 * Chow test at the specified transition 2024-07 rejects stability (p=0.0000, level shift -27.16).
 * sup-Wald locates its most likely break at 2019-06 (bootstrap p=0.0033, significant); that is outside 6 months of the specified transition 2024-07.
 * NOTE: the specified transition 2024-07 lies outside the trimmed search window (1985-04..2019-06), so the sup-Wald search could not have selected it. A break was not found there because it was never a candidate — that is not evidence against Hypothesis C. Re-run with a smaller trim, or rely on the specified-date Chow test, which can test a date the search cannot reach.
 * The gap is 4.7x more level than month-to-month variation (|mean| / sd), which is the shape a collection-mode shift would produce.
 * No usable independently collected series, so the decisive falsification condition could not be applied.
 * The specified-date Chow test rejects stability, which is consistent with C but equally consistent with any same-dated shock.


### A methodological finding: a comparison series that contains Michigan

The obvious comparison series is FRED `USACSCICP02STSAM`, the OECD consumer
confidence indicator for the United States. That series correlates 1.0000 with
the Michigan survey after 2010, in levels and in differences, because the OECD
consumer input for the United States **is** the Michigan survey. A test against
it gives a gap ratio near 1.00 and "rejects" Hypothesis C with Michigan's own
numbers. The result looks clean and it is circular.

The module now screens every candidate. The screen rejects a candidate above
0.99 in levels **and** above 0.90 in differences.

In [14]:
for candidate in sd.COMPARISON_CANDIDATES:
    mark = "usable   " if candidate.independent_of_michigan else "REJECTED "
    print(f"{mark} {candidate.series_id:<20} {candidate.label}")
    print(f"           {candidate.note}")
    print()

usable    FMNHSHPSIUS          Fannie Mae National Housing Survey: Home Purchase Sentiment Index
           Independently fielded consumer survey (~1,000 households/month). DISCONTINUED after 2025-09, so it overlaps only the first nine holdout months, and it is housing-weighted rather than a general confidence index.

usable    BSCICP02USM460S      OECD Business Tendency Surveys: manufacturing confidence, United States
           Covers the whole holdout and is collected from firms, not households, so it is independent of any consumer-survey mode change — but it measures business rather than consumer confidence, which is a different construct.

REJECTED  CSCICP03USM665S      OECD composite consumer confidence, United States (amplitude adjusted)
           The series ID in BRAINSTORMING.md. It EXISTS on FRED but its last observation is 2024-01: it is stale/discontinued and covers none of the 2025-26 holdout.

REJECTED  USACSCICP02STSAM     OECD consumer opinion surveys: composite consum

## Step 6. Hypothesis B. Not yet tested

The media corpus is still downloading. The cell below gives the state of the
corpus, and the exact list of results that the corpus blocks. Nothing in this
step is estimated. A missing result and a negative result are different
findings.

In [15]:
days = br.count_cached_media_days(ROOT / "data" / "raw" / "gdelt")
print(f"GDELT days cached: {days} of {br.GDELT_TARGET_DAYS} "
      f"(the minimum for a monthly series is {br.GDELT_MIN_DAYS})")
print(f"media series available: {ctx.media_available}")
print()
print("Blocked by the missing corpus:")
for item in (
    "Model 3 in Table 3 — specified, not estimated",
    "Table 2, the SVD topics and term loadings",
    "Table 4, Panel A, the Granger F-tests",
    "Table 4, Panel B, the Rigobon-Sack IV",
):
    print("  -", item)

GDELT days cached: 4 of 232 (the minimum for a monthly series is 200)
media series available: False

Blocked by the missing corpus:
  - Model 3 in Table 3 — specified, not estimated
  - Table 2, the SVD topics and term loadings
  - Table 4, Panel A, the Granger F-tests
  - Table 4, Panel B, the Rigobon-Sack IV


In [16]:
display(Markdown(br.build_table2(ctx).to_markdown()))
display(Markdown(br.build_table4(ctx).to_markdown()))

**Table 2: SVD latent economic topics and term loadings**

| Topic | Term | Loading (U_k) |
|:---|---:|---:|
| n/a | n/a - the GDELT corpus is still downloading: 4 of 232 days are cached, so no TF-IDF matrix and no SVD basis exist yet | n/a |

*Notes:* Loadings are entries of the left singular vectors U_k of the TF-IDF term-document matrix, which is not mean-centred.  

**Table 4: Granger causality tests and Rigobon-Sack IV transmission estimates**

*Panel A. Granger causality F-tests*

| Direction | Lags | F | p | n |
|:---|---:|---:|---:|---:|
| n/a - no media series to test; the GDELT corpus is still downloading: 4 of 232 days are cached, so no TF-IDF matrix and no SVD basis exist yet | n/a | n/a | n/a | n/a |

*Panel B. Rigobon-Sack heteroskedasticity IV*

| Parameter | Estimate | SE | p | n |
|:---|---:|---:|---:|---:|
| n/a - the Rigobon-Sack system needs a media series; the GDELT corpus is still downloading: 4 of 232 days are cached, so no TF-IDF matrix and no SVD basis exist yet | n/a | n/a | n/a | n/a |

*Notes:* Granger tests report predictive precedence, not causation: a common driver of both coverage and sentiment can produce them with no causal channel. The IV is attempted for exactly that reason.  
Standard errors are Newey-West HAC with maxlags=12.  
Significance: * p<0.10, ** p<0.05, *** p<0.01.  
The reverse direction of the Rigobon-Sack system is not identified. The system is short by one restriction. Only media -> gap is estimated. Refer to paragraph 7.3.  

**A correction to the brief on Rigobon-Sack.** The brief gives an order
condition `N^2 - N - 2K > 0`. With `N = 2` this is `2 - 2K`, which holds only
when `K = 0`. The formula is uninformative for a two-variable system. The brief
also asks for `det(Omega_H - Omega_L) != 0`. Under the identifying assumption
that one shock's variance shifts, that matrix has rank 1 and its determinant is
exactly zero. A determinant near zero therefore **agrees** with identification.
The operative test is different: the media series' own variance must differ
across the regimes.

The system is short by one restriction. The study estimates the direction
media to gap only. The reverse direction is not identified and is not reported.

**FinBERT, measured and not quoted.** A probe set holds 30 macro sentences,
with 10 hawkish, 10 dovish and 10 neutral. FinBERT scores 0.300 against 0.333
for chance. It calls 6 of 10 hawkish sentences dovish, and 7 of 10 dovish
sentences hawkish. FinBERT learns from market text, which asks "is this good for
the asset?". Macro policy text asks a different question. The probe needs the
`transformers` model weights, so this notebook does not download them. The
module `nlp_media_svd.finbert_probe` runs the probe.

## Step 7. The report, assembled

`assemble_sections` is a pure function. It turns the context into sections. It
writes no file. The checks below run on the sections, so they need no PDF.

In [17]:
sections = br.assemble_sections(ctx)
for section in sections:
    missing = len(section.missing_results())
    tables = len(section.tables())
    flag = f"  [{missing} not-available block(s)]" if missing else ""
    print(f"{br.estimate_section_lines(section)/br.LINES_PER_PAGE:5.2f} page  "
          f"{section.heading[:58]:<58}{flag}")
print()
print(f"estimated pages: {br.estimate_pages(sections):.1f}")
print(f"placeholder markers found: {br.find_placeholder_markers(sections)}")

 0.57 page  The 2026 Sentiment Gap: summary                           
 0.76 page  1. The data and its limits                                  [1 not-available block(s)]
 1.62 page  2. The baseline macro model                               
 1.41 page  3. The Sentiment Gap                                        [1 not-available block(s)]
 1.06 page  4. Hypothesis A. Economic pressure that the baseline misse
 1.45 page  5. Hypothesis B. Media selection and framing: not yet test  [1 not-available block(s)]
 1.21 page  6. Hypothesis C. The survey method changed                  [1 not-available block(s)]
 1.23 page  7. Findings about method                                  
 0.64 page  8. What is not yet available                              
 1.08 page  9. Provenance and reproduction                            

estimated pages: 12.0
placeholder markers found: []


In [18]:
print("Every not-available block in the report:\n")
for section in sections:
    for missing in section.missing_results():
        print(f"[{section.number or 'summary'}] {missing.item}")
        print(f"    cause: {missing.cause}")
        for blocked in missing.blocks:
            print(f"    blocks: {blocked}")
        print()

Every not-available block in the report:

[1] The FRED series PCEPI, MORTGAGE30US and TDSP
    cause: this build has no usable FRED key, so no FRED request was made. The panel used the documented approximation of the price level in place of PCEPI. The approximation accumulates the published year-on-year rates in logs.
    blocks: Live estimation of Model 2. Table 3 prints the recorded Model 2 numbers and marks them as recorded.
    blocks: The cross-survey leg of Hypothesis C.

[3] The gap in 2025-10, 2026-09, 2026-10
    cause: one regressor or more has no published value in each of these months: 2025-10: missing unemployment; 2026-09: missing pce_yoy; 2026-10: missing unemployment, pce_yoy, real_gdp_qoq
    blocks: These months have no row in Table 1, Panel C, and no bar in Figure 2. The sentiment column itself continues past the last gap month.

[5] Every result for Hypothesis B
    cause: the media corpus is not complete, because the GDELT corpus is still downloading: 4 of 232 days

## Step 8. The register of cross-checked constants

Most numbers in the report come from the live modules. A small set cannot be
recomputed in every environment, because FRED needs a key and a network, or
because a second tool did the check. Each one has a tag and a source.

In [19]:
register = pd.DataFrame(
    [
        {
            "tag": br.CONSTANT_TAGS[item.key],
            "quantity": item.label,
            "value": item.value,
            "source": item.source,
        }
        for item in br.VERIFIED_CONSTANTS
    ]
)
display(register.style.hide(axis="index"))

tag,quantity,value,source
C1,Share of the gap that Hypothesis A absorbs,31%,BRAINSTORMING.md paragraph 2.1 and paragraph 7
C2,Model 2 R-squared,0.835,BRAINSTORMING.md paragraph 2.1
C3,Model 2 AIC,1117.4,BRAINSTORMING.md paragraph 2.1
C4,Model 2 mean out-of-sample gap,-17.40,BRAINSTORMING.md paragraph 2.1
C5,Mortgage rate coefficient in Model 2,"-2.936, p < 0.001",BRAINSTORMING.md paragraph 2.1
C6,Price level p-value in Model 2,0.310,BRAINSTORMING.md paragraph 2.1
C7,Variance inflation factor of the price level,3.73,BRAINSTORMING.md paragraph 2.1
C8,Chow F at the 2024-07 transition,81.3,"BRAINSTORMING.md paragraph 2.3, recorded run"
C9,Level shift at the 2024-07 transition,-35.27,"PROGRESS.md paragraph 7.4, independent recomputation"
C10,Supremum-Wald break date,1990-07 (module) / 1990-08 (independent path),PROGRESS.md paragraph 7.4


## Step 9. Build the PDF

The cell below is not run by default, because it writes a file. Remove the
comment to build the report.

In [20]:
# path, pages, sections = br.build()
# print(path, pages)
print("To build the report, run: .venv/bin/python build_report.py")

To build the report, run: .venv/bin/python build_report.py


## What this notebook commits to

1. The gap is about -25 on average and about -33 at its worst. **18 of 19
   out-of-sample months fall outside the 95% prediction interval.** This count
   is the quantity to explain.
2. Hypothesis A absorbs about 31% of the gap. The mortgage rate does the work.
   The price level is not separately identified, so it gets no causal reading.
3. Hypothesis C is supported in part. The ratio is 0.258 and the threshold is
   0.25. Print both together.
4. Hypothesis B is **not tested**. Its falsification conditions are fixed and
   are given in Step 6.
5. Each causal statement carries the strength of its identification:
   association, predictive precedence, or instrumented.